# Logistic Regression - accidentologie (3 runs)

Objectif:
- construire la cible de gravite selon la regle metier
- entrainer 3 runs de regression logistique avec variation d'hyperparametres
- comparer les metriques
- preparer les artefacts et l'integration MLflow pour la suite

## Regle metier cible

- **Non grave (0)**: non hospitalisation ou hospitalisation de moins de 24h
- **Grave (1)**: hospitalisation de plus de 24h ou deces

Implementation:
- si `grave` (0/1) existe, on l'utilise directement
- sinon, fallback sur `grav_max_rank` (grave = 1 si `grav_max_rank >= 2`)
- sinon, fallback sur `grav` BAAC (2/3 -> grave, 1/4 -> non grave)

In [1]:
from __future__ import annotations

import json
from datetime import UTC, datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def find_project_root(marker: str = "out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    return Path.cwd()


ROOT = find_project_root("out")
DATA_PATH = ROOT / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
ARTIFACT_DIR = ROOT / "out" / "logreg_experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "grave"
SEED = 42
CV_SPLITS = 3
N_ITER = 24
THRESHOLD = 0.5
MAX_ROWS = None  # ex: 50000 pour accelerer

# Memes 15 features que les autres notebooks pour comparaison equitable
PRODUCT15_V2 = [
    "dep",
    "lum",
    "atm",
    "catr",
    "agg",
    "int",
    "circ",
    "col",
    "vma_bucket",
    "catv_family_4",
    "manv_mode",
    "driver_age_bucket",
    "choc_mode",
    "driver_trajet_family",
    "time_bucket",
]

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

ROOT: /home/maxime/simplonalternance/alternance-CICDprediction
DATA_PATH: /home/maxime/simplonalternance/alternance-CICDprediction/out/accidents_model_ready_kept_with_time_bucket.csv
ARTIFACT_DIR: /home/maxime/simplonalternance/alternance-CICDprediction/out/logreg_experiments


In [2]:
assert DATA_PATH.exists(), f"Fichier introuvable: {DATA_PATH}"
df = pd.read_csv(DATA_PATH, sep=";", low_memory=False, nrows=MAX_ROWS)

assert TARGET in df.columns, f"Colonne cible absente: {TARGET}"
missing_feats = [c for c in PRODUCT15_V2 if c not in df.columns]
assert not missing_feats, f"Colonnes manquantes dans le CSV: {missing_feats}"

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df[PRODUCT15_V2].copy()
y = df[TARGET].copy()

print("Shape:", X.shape)
print("Taux grave=1:", round(float(y.mean()), 4))
print("Features utilisees:", list(X.columns))

Shape: (164526, 15)
Taux grave=1: 0.3608
Features utilisees: ['dep', 'lum', 'atm', 'catr', 'agg', 'int', 'circ', 'col', 'vma_bucket', 'catv_family_4', 'manv_mode', 'driver_age_bucket', 'choc_mode', 'driver_trajet_family', 'time_bucket']


## Preprocessing

- Split train/val/test stratifie (60/20/20)
- **val** sert a optimiser le seuil, **test** sert a evaluer les metriques finales
- standardisation numerique + one-hot encoding categoriel
- Les 15 features sont toutes categorielles

In [3]:
# Split 60/20/20 : train / val (seuil) / test (evaluation finale)
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp,
)

# Toutes les features product15_v2 sont categorielles
cat_cols = PRODUCT15_V2[:]
num_cols = []

num_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

transformers = [("cat", cat_pipe, cat_cols)]
if num_cols:
    transformers.insert(0, ("num", num_pipe, num_cols))

preprocess = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Features categorielles:", len(cat_cols))
print("Features numeriques:", len(num_cols))

Train: (98715, 15) | Val: (32905, 15) | Test: (32906, 15)
Features categorielles: 15
Features numeriques: 0


## 3 runs Logistic Regression

- `logreg_auc_opt` optimise `roc_auc`
- `logreg_f1_opt` optimise `f1`
- `logreg_recall_opt` optimise `recall`

In [4]:
def evaluate_binary(
    y_true: pd.Series, proba: np.ndarray, threshold: float = THRESHOLD
) -> dict[str, float]:
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
    }


def best_threshold_by_f1(
    y_true: pd.Series, proba: np.ndarray
) -> tuple[float, dict[str, float]]:
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = [evaluate_binary(y_true, proba, float(t)) for t in thresholds]
    df_thr = pd.DataFrame(rows)
    idx = int(df_thr["f1"].idxmax())
    best = df_thr.loc[idx].to_dict()
    return float(best["threshold"]), {k: float(v) for k, v in best.items()}


def make_search(scoring: str, param_dist) -> RandomizedSearchCV:
    model = LogisticRegression(random_state=SEED, n_jobs=None)
    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)
    return RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring=scoring,
        n_jobs=-1,
        cv=cv,
        random_state=SEED,
        verbose=1,
        refit=True,
    )


PARAM_AUC = [
    {
        "model__penalty": ["l2"],
        "model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
        "model__solver": ["lbfgs", "saga"],
        "model__class_weight": [None, "balanced"],
        "model__max_iter": [2000, 4000],
    }
]

PARAM_F1 = [
    {
        "model__penalty": ["l1"],
        "model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0],
        "model__solver": ["liblinear", "saga"],
        "model__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}],
        "model__max_iter": [3000, 5000],
    },
    {
        "model__penalty": ["elasticnet"],
        "model__C": [0.01, 0.03, 0.1, 0.3, 1.0],
        "model__solver": ["saga"],
        "model__l1_ratio": [0.2, 0.4, 0.6, 0.8],
        "model__class_weight": ["balanced", {0: 1, 1: 2}],
        "model__max_iter": [4000, 7000],
    },
    {
        "model__penalty": ["l2"],
        "model__C": [0.03, 0.1, 0.3, 1.0, 3.0],
        "model__solver": ["lbfgs", "saga"],
        "model__class_weight": ["balanced", {0: 1, 1: 2}],
        "model__max_iter": [2500, 4000],
    },
]

PARAM_RECALL = [
    {
        "model__penalty": ["l2"],
        "model__C": [0.01, 0.03, 0.1, 0.3, 1.0],
        "model__solver": ["lbfgs", "saga"],
        "model__class_weight": ["balanced", {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}],
        "model__max_iter": [3000, 5000, 7000],
    },
    {
        "model__penalty": ["elasticnet"],
        "model__C": [0.01, 0.03, 0.1, 0.3, 1.0],
        "model__solver": ["saga"],
        "model__l1_ratio": [0.1, 0.2, 0.4, 0.6],
        "model__class_weight": ["balanced", {0: 1, 1: 3}, {0: 1, 1: 4}],
        "model__max_iter": [5000, 8000],
    },
]

EXPERIMENTS = [
    ("logreg_auc_opt", "roc_auc", PARAM_AUC),
    ("logreg_f1_opt", "f1", PARAM_F1),
    ("logreg_recall_opt", "recall", PARAM_RECALL),
]

In [5]:
trained = {}
rows = []

for run_name, scoring, param_dist in EXPERIMENTS:
    print(f"\n=== {run_name} | scoring={scoring} ===")
    search = make_search(scoring=scoring, param_dist=param_dist)
    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # Seuil optimise sur val (pas sur test)
    proba_val = best_model.predict_proba(X_val)[:, 1]
    best_thr, _ = best_threshold_by_f1(y_val, proba_val)

    # Metriques finales sur test (jamais vu par le modele ni le seuil)
    proba_test = best_model.predict_proba(X_test)[:, 1]
    metrics_05 = evaluate_binary(y_test, proba_test, threshold=THRESHOLD)
    metrics_best = evaluate_binary(y_test, proba_test, threshold=best_thr)

    trained[run_name] = {
        "search": search,
        "model": best_model,
        "proba_test": proba_test,
        "metrics_05": metrics_05,
        "best_threshold": best_thr,
        "metrics_best": metrics_best,
    }

    rows.append(
        {
            "run_name": run_name,
            "optimized_for": scoring,
            "cv_best_score": float(search.best_score_),
            "f1_05": metrics_05["f1"],
            "roc_auc_05": metrics_05["roc_auc"],
            "f1_best": metrics_best["f1"],
            "best_threshold_f1": metrics_best["threshold"],
            "best_params": search.best_params_,
        }
    )

results_df = (
    pd.DataFrame(rows)
    .sort_values(["f1_best", "roc_auc_05"], ascending=False)
    .reset_index(drop=True)
)
results_df[
    [
        "run_name",
        "optimized_for",
        "cv_best_score",
        "f1_05",
        "roc_auc_05",
        "f1_best",
        "best_threshold_f1",
    ]
]


=== logreg_auc_opt | scoring=roc_auc ===
Fitting 3 folds for each of 24 candidates, totalling 72 fits


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_m


=== logreg_f1_opt | scoring=f1 ===
Fitting 3 folds for each of 24 candidates, totalling 72 fits


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_


=== logreg_recall_opt | scoring=recall ===
Fitting 3 folds for each of 24 candidates, totalling 72 fits


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/linear_m

,run_name,optimized_for,cv_best_score,f1_05,roc_auc_05,f1_best,best_threshold_f1
0,logreg_f1_opt,f1,0.670374,0.672631,0.806853,0.672631,0.50
1,logreg_auc_opt,roc_auc,0.803754,0.671671,0.806825,0.671965,0.47
2,logreg_recall_opt,recall,0.910424,0.635543,0.803087,0.667171,0.69


In [6]:
registry_rows = []

for row in rows:
    run_name = row["run_name"]
    payload = trained[run_name]

    model = payload["model"]
    proba_test = payload["proba_test"]
    best_thr = payload["best_threshold"]

    model_path = ARTIFACT_DIR / f"{run_name}.joblib"
    pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
    meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

    joblib.dump(model, model_path)

    pred_df = pd.DataFrame(
        {
            "y_true": y_test.to_numpy(),
            "proba": proba_test,
            "pred_05": (proba_test >= THRESHOLD).astype(int),
            "pred_best_f1": (proba_test >= best_thr).astype(int),
        }
    )
    pred_df.to_csv(pred_path, index=False)

    meta = {
        "run_name": run_name,
        "optimized_for": row["optimized_for"],
        "dataset": str(DATA_PATH),
        "target": TARGET,
        "seed": SEED,
        "cv_splits": CV_SPLITS,
        "n_iter": N_ITER,
        "cv_best_score": row["cv_best_score"],
        "threshold_05": THRESHOLD,
        "best_threshold_f1": best_thr,
        "metrics_05": payload["metrics_05"],
        "metrics_best_f1": payload["metrics_best"],
        "best_params": payload["search"].best_params_,
        "model_path": str(model_path),
        "predictions_path": str(pred_path),
        "created_at_utc": datetime.now(UTC).isoformat(),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    registry_rows.append(
        {
            "run_name": run_name,
            "model_path": str(model_path),
            "predictions_path": str(pred_path),
            "meta_path": str(meta_path),
        }
    )

registry_df = pd.DataFrame(registry_rows)
summary_df = results_df.merge(registry_df, on="run_name", how="left")
summary_df

,run_name,optimized_for,cv_best_score,f1_05,roc_auc_05,f1_best,best_threshold_f1,best_params,model_path,predictions_path,meta_path
0,logreg_f1_opt,f1,0.670374,0.672631,0.806853,0.672631,0.50,"{'model__solver': 'liblinear', 'model__penalty...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
1,logreg_auc_opt,roc_auc,0.803754,0.671671,0.806825,0.671965,0.47,"{'model__solver': 'lbfgs', 'model__penalty': '...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
2,logreg_recall_opt,recall,0.910424,0.635543,0.803087,0.667171,0.69,"{'model__solver': 'saga', 'model__penalty': 'e...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...


## Integration MLflow (desactivee par defaut)

Bloc prepare pour la prochaine etape.

In [ ]:
ENABLE_MLFLOW = True
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "accidentologie_model_benchmark"
ENABLE_MODEL_REGISTRY = True
REGISTERED_MODEL_NAME = "briefml-logreg-product15-v2-time-bucket"

if ENABLE_MLFLOW:
    import mlflow
    import mlflow.sklearn
    import numpy as np

    def _to_params(d):
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(
                    v,
                    (
                        int,
                        float,
                        str,
                        bool,
                        np.integer,
                        np.floating,
                        np.bool_,
                    ),
                ):
                    out[k] = v.item() if hasattr(v, "item") else v
        return out

    def _normalize_metrics(d, prefix="valid_"):
        keys = {
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "threshold",
        }
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if k in keys and isinstance(
                    v,
                    (int, float, np.integer, np.floating),
                ):
                    out[f"{prefix}{k}"] = float(v)
        return out

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    # On enregistre le meilleur run (par f1_best)
    best_run_name = results_df.loc[results_df["f1_best"].idxmax(), "run_name"]

    for row in rows:
        run_name = row["run_name"]
        payload = trained.get(run_name, {})
        search = payload.get("search")
        model = payload.get("model")

        if search is None or model is None:
            print(f"[mlflow] skip {run_name}: search/model manquant")
            continue

        metrics_05 = _normalize_metrics(payload.get("metrics_05"), prefix="valid_")
        metrics_best = _normalize_metrics(
            payload.get("metrics_best"),
            prefix="valid_bestf1_",
        )

        model_path = ARTIFACT_DIR / f"{run_name}.joblib"
        pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
        meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

        # Registry uniquement pour le meilleur run
        register_name = (
            REGISTERED_MODEL_NAME
            if (ENABLE_MODEL_REGISTRY and run_name == best_run_name)
            else None
        )

        with mlflow.start_run(run_name=run_name):
            mlflow.set_tags(
                {
                    "notebook": "15_logistic_regression",
                    "model_family": "logistic_regression",
                    "model_flavor": "sklearn",
                    "tag": "accidentologie",
                    "optimized_for": str(row.get("optimized_for", "unknown")),
                }
            )
            if register_name:
                mlflow.set_tag("registry_enabled", "true")

            mlflow.log_param("seed", int(SEED))
            mlflow.log_param("cv_splits", int(CV_SPLITS))
            mlflow.log_param("n_iter", int(N_ITER))
            mlflow.log_param("target", TARGET)
            mlflow.log_param(
                "cv_primary_metric",
                str(row.get("optimized_for", "unknown")),
            )
            mlflow.log_params(_to_params(search.best_params_))
            mlflow.log_metric(
                "cv_primary_score",
                float(search.best_score_),
            )

            if metrics_05:
                mlflow.log_metrics(metrics_05)
            if metrics_best:
                mlflow.log_metrics(metrics_best)

            model_info = mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                registered_model_name=register_name,
            )

            for p, art in [
                (model_path, "models"),
                (pred_path, "predictions"),
                (meta_path, "metadata"),
            ]:
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path=art)

            print(f"[mlflow] logged: {run_name}")
            if register_name and getattr(
                model_info,
                "registered_model_version",
                None,
            ):
                ver = model_info.registered_model_version
                print(f"[mlflow] model registered: {register_name} v{ver}")
else:
    print(
        "MLflow desactive. "
        "Passe ENABLE_MLFLOW=True pour logger "
        "les runs Logistic Regression."
    )